# Tropical geometry: a network *is* a piecewise-linear map

A feed-forward net with ReLU / max-pool / morphological layers computes a **tropical
rational map** — a difference of two tropical polynomials, so convex-piecewise-linear
per piece. Its input space splits into **linear regions** (affine inside each), its
**decision boundary** is a tropical hypersurface, and its expressivity is measured by
how many regions it can carve. `tikitorch.geometry` measures all three on a real model.


In [ ]:
# --- Setup: install TikiTorch (run once) ---
# TikiTorch is research code, not on PyPI. In Colab, install from the repo.
try:
    import tikitorch  # already installed?
except ImportError:
    REPO_URL = "https://github.com/nyuolab/tikitorch.git"
    import subprocess, sys, os
    if not os.path.exists("tikitorch_repo"):
        subprocess.run(["git", "clone", REPO_URL, "tikitorch_repo"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "tikitorch_repo"], check=False)
    import tikitorch
import torch
print("TikiTorch", getattr(tikitorch, "__version__", "?"), "| torch", torch.__version__)


## Linear regions = distinct local Jacobians

Inside one linear region the map is affine, so its input→output Jacobian is constant;
crossing into another region changes it. `jacobian_signature` returns that (normalized)
gradient as a region fingerprint, and `count_linear_regions` counts the uniques over a
sample. We use a **hard** tropical net (`use_softmin=False`) — the genuinely
piecewise-linear regime (a smooth/soft net has a single, ever-changing Jacobian).


In [ ]:
import torch, tikitorch as tt
from tikitorch.geometry import count_linear_regions, jacobian_signature

torch.manual_seed(0)
net = tt.models.TropicalMLP(input_dim=2, d_model=32, n_layers=1, num_classes=3,
                            use_softmin=False)            # hard => piecewise-linear
pts = torch.randn(800, 2)
print("distinct linear regions over 800 samples:", count_linear_regions(net, pts))
print("region fingerprint at a point:", jacobian_signature(net, torch.tensor([0.3, -1.1])))


## More units → more pieces (then random init saturates)

Width buys linear regions — each tropical unit can add a fold. At *random* init this
saturates, and very deep stacks can even **collapse** (routing funnels through a few
cells) — which is why identity/residual init matters at depth, and why *training* carves
far more regions than random init does.


In [ ]:
for W in [4, 8, 16, 32]:
    m = tt.models.TropicalMLP(input_dim=2, d_model=W, n_layers=1, num_classes=3,
                              use_softmin=False)
    print(f"  width={W:3d}  ->  {count_linear_regions(m, pts):4d} regions")


## See the decision regions (the tropical hypersurface)

`decision_regions_2d` returns `(preds[n,n], xs, ys)` — the predicted class over a grid.
The borders between classes are the network's decision boundary: straight polytope
faces, not smooth curves.


In [ ]:
from tikitorch.geometry import decision_regions_2d
preds, xs, ys = decision_regions_2d(net, xrange=(-3, 3), yrange=(-3, 3), n=120)
print("grid:", tuple(preds.shape), "classes:", sorted(set(preds.flatten().tolist())))
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(4, 4))
    plt.imshow(preds, origin="lower", extent=[-3, 3, -3, 3], cmap="tab10", aspect="auto")
    plt.title("TropicalMLP decision regions (polytope arrangement)")
    plt.xlabel("x0"); plt.ylabel("x1"); plt.show()
except ImportError:
    print("(install matplotlib to see the region map)")


## Newton polytopes

A tropical polynomial `max_k (c_k + <a_k, x>)` has Newton polytope `conv{a_k}`; its
**vertices are the monomials active somewhere** — i.e. the linear pieces. `newton_polytope`
returns those vertices (uses SciPy's ConvexHull when available; interior monomials are
dropped because they never become a separate linear piece).


In [ ]:
from tikitorch.geometry import newton_polytope
exps = torch.tensor([[0., 0.], [2., 0.], [0., 2.], [1., 1.]])   # (1,1) is interior
verts = newton_polytope(exps)
print("monomial exponents:", exps.tolist())
print("Newton-polytope vertices:", [v for v in verts.tolist()])


## Why it matters

- **Expressivity** — counting regions is the standard "how complex can this be" yardstick;
  tropical geometry gives the cleanest bounds and `count_linear_regions` checks them.
- **Certification** — affine-per-region means monotonicity / Lipschitz / robustness can be
  reasoned about *exactly* per region.
- **Pruning** — interior monomials (no new vertex) are redundant; see `tikitorch.prune`.

Concept companion: **Guide → Tropical geometry of neural nets**.
